# 03 - Pairwise judging with a Kaggle-hosted model

Run this notebook after notebook 02 has completed. Nothing has to be edited.

1. **Create and import:** open https://www.kaggle.com/benchmarks/tasks/new so that `kaggle_benchmarks` and the
   Kaggle model proxy (`MODEL_PROXY_URL`, `MODEL_PROXY_API_KEY`) are available, then **File -> Import Notebook ->**
   upload this file.
2. **Settings:** no accelerator is needed. Internet is not needed for the code: it is copied from the attached input.
3. **Inputs:** **Add Input -> Notebooks ->** the saved, completed version of notebook 02.
4. **Run:** **Save Version -> Save & Run All (Commit)**.
5. **Afterwards:** send the status of the version and its log; the last cells print one line per comparison, the
   first verdicts with their reasons, the judgment tables and whether any pair is unresolved.

Nothing is generated here. The notebook reads the saved pairwise JSONL files, judges 5 pairs per request, waits at
least 5 seconds between all request attempts, tries a failing batch up to 3 times, and appends results after every
batch. Each comparison starts with a one-line availability check, so an unavailable judge model raises before any
pair is sent. A missing pair file raises before the first request.

## 1. Get the code (copied from the attached notebook 02 output; cloned from GitHub only if none is attached)

In [ ]:
import os, shlex, shutil, subprocess, sys
from pathlib import Path

# ---- Settings: the defaults work as they are, nothing has to be edited ----------------------
REPO_URL = "https://github.com/zainahmadlums/genai-pa2.git"
REPO_BRANCH = "main"
CODE_DATASET_DIR = ""   # optional: use an uploaded copy of the code, e.g. "/kaggle/input/my-pa2-code/genaiPA2"
PREFER_ATTACHED_CODE = True   # True: take the code from an attached notebook output instead of GitHub
# ---------------------------------------------------------------------------------------------

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO_DIR = WORK / "small-model-alignment"
CODE_MARKER = "scripts/judge_pairwise_kaggle.py"   # present only in a complete copy of the code


def run(*argv, **popen_options):
    """Run one command given as separate arguments (no shell, so spaces in paths are safe), output shown live.

    A non-zero exit status raises CalledProcessError, which stops "Run All" at this cell.
    """
    argv = [str(arg) for arg in argv]
    print("$", shlex.join(argv), flush=True)
    with subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", **popen_options) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, argv)


def py(*argv, **popen_options):
    """Run a script or module with this notebook's Python interpreter."""
    run(sys.executable, *argv, **popen_options)


def find_code_in_inputs():
    """A copy of the code inside an attached input (an uploaded dataset or an earlier notebook's output)."""
    matches = sorted(INPUT.glob("**/" + CODE_MARKER))
    return matches[0].parents[1] if matches else None


if (REPO_DIR / CODE_MARKER).exists():
    print("using the code already at", REPO_DIR, "as it is (an existing copy is never updated)")
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a complete copy of the code. Delete it (or restart the "
                       "session) and run this cell again.")
else:
    if CODE_DATASET_DIR:
        source = Path(CODE_DATASET_DIR)
        if not (source / CODE_MARKER).exists():
            raise FileNotFoundError(f"CODE_DATASET_DIR={CODE_DATASET_DIR!r} does not contain {CODE_MARKER}")
    else:
        source = find_code_in_inputs() if PREFER_ATTACHED_CODE else None
    if source is not None:
        shutil.copytree(source, REPO_DIR, ignore=shutil.ignore_patterns("outputs", ".git", ".venv", "__pycache__"))
        print("copied the code from", source)
    else:
        if PREFER_ATTACHED_CODE:
            print("no copy of the code in the attached inputs; cloning it instead (needs Internet On)")
        run("git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR)

os.chdir(REPO_DIR)
os.environ["PYTHONUNBUFFERED"] = "1"   # training / generation logs appear line by line
sys.path.insert(0, str(REPO_DIR))
print("working directory:", os.getcwd())

In [ ]:
# Judging needs only the standard library plus kaggle_benchmarks (or openai for the proxy fallback).
import importlib
for name in ("kaggle_benchmarks", "openai"):
    try:
        module = importlib.import_module(name)
        print(name, getattr(module, "__version__", "installed"))
    except Exception as exc:
        print(name, "NOT AVAILABLE:", type(exc).__name__, exc)

## 2. Collect the pairwise files (and earlier judgments when resuming in a new session)

In [ ]:
PAIRS_DIR = Path("outputs/eval/pairs")
JUDGED_DIR = Path("outputs/eval/judged")
PAIRS_DIR.mkdir(parents=True, exist_ok=True)
JUDGED_DIR.mkdir(parents=True, exist_ok=True)

for source in sorted(INPUT.glob("**/outputs/eval/pairs/*_pairs.jsonl")):
    target = PAIRS_DIR / source.name
    if not target.exists():
        shutil.copyfile(source, target)
# Judgments from an earlier version of THIS notebook (attach that version as input to continue it).
for source in sorted(INPUT.glob("**/outputs/eval/judged/*")):
    target = JUDGED_DIR / source.name
    if source.is_file() and not target.exists():
        shutil.copyfile(source, target)

REQUIRED = ["dpo_uf_vs_sft", "dpo_hh_vs_sft", "persona_vs_base_style", "persona_vs_base_quality"]
print("pairs:", sorted(path.name for path in PAIRS_DIR.glob("*_pairs.jsonl")))
print("existing judgment files:", sorted(path.name for path in JUDGED_DIR.glob("*_judged.jsonl")))
missing = [name for name in REQUIRED if not (PAIRS_DIR / f"{name}_pairs.jsonl").exists()]
if missing:
    raise FileNotFoundError(f"no pair file for {missing}: attach the saved, completed version of notebook 02 "
                            "(Add Input -> Notebooks) and run again")

## 3. Choose the judge model

`google/gemini-3.6-flash` is the model suggested by the assignment. The cell lists what this notebook actually
exposes. If the suggested model is not available (the availability check in section 4 raises), set `JUDGE_MODEL` to
a listed model and name it in the report.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "scripts"))
import judge_pairwise_kaggle as judge_cli

JUDGE_MODEL = "google/gemini-3.6-flash"
BACKEND = "auto"   # kaggle_benchmarks first, then the OpenAI-compatible model proxy
# Saved judgments are reused only for the same backend, judge model and batch size. If you change one of them
# after judging has started, the run stops before sending anything. Set this to True to rename the old result
# files to *.stale<N> and judge every pair again with the new settings.
INVALIDATE_INCOMPATIBLE = False
judge_cli.main(["--list_models"])

In [ ]:
def judge(comparison):
    """Judge one comparison. A missing pair file or a judge that cannot be used raises instead of being skipped."""
    pairs_path = PAIRS_DIR / f"{comparison}_pairs.jsonl"
    if not pairs_path.exists():
        raise FileNotFoundError(f"{pairs_path} is missing")
    argv = ["--pairs_path", str(pairs_path),
            "--output_path", str(JUDGED_DIR / f"{comparison}_judged.jsonl"),
            "--judge_model", JUDGE_MODEL, "--backend", BACKEND,
            "--batch_size", "5", "--sleep_seconds", "5", "--max_retries", "3", "--reuse_judgments"]
    if INVALIDATE_INCOMPATIBLE:
        argv += ["--invalidate_incompatible"]
    try:
        return judge_cli.main(argv)
    except SystemExit as exc:
        raise RuntimeError(f"judging {comparison} stopped: {exc}") from exc

## 4. Judging

About 60 requests per 300-pair comparison. If a cell stops (quota, disconnect), run it again: finished pairs are
skipped and failed ones are retried.

In [ ]:
judge("dpo_uf_vs_sft")

In [ ]:
judge("dpo_hh_vs_sft")

In [ ]:
judge("persona_vs_base_style")
judge("persona_vs_base_quality")

In [ ]:
# Optional: SFT vs base (own analysis only).
RUN_OPTIONAL = False
if RUN_OPTIONAL:
    judge("sft_vs_base")

## 5. Summaries

In [ ]:
import json
from alignment.io_utils import read_jsonl

for path in sorted(JUDGED_DIR.glob("*.summary.json")):
    summary = json.loads(path.read_text(encoding="utf-8"))
    print(f"{summary['comparison']}: judge={summary['judge_model']} pairs={summary['total_pairs']} "
          f"wins={summary['candidate_wins']} losses={summary['baseline_wins']} ties={summary['ties']} "
          f"unknown={summary['unknown']} failures={summary['failures']} complete={summary['complete']} "
          f"win_rate_excl_ties={summary['candidate_win_rate_excluding_ties']}")
# The first three verdicts and reasons of every comparison, to see that the judge answers sensibly.
for path in sorted(JUDGED_DIR.glob("*_judged.jsonl")):
    for record in read_jsonl(path)[:3]:
        print(f"  {path.name} {record['prompt_id']} | verdict: {record['verdict']} | winner: "
              f"{record['winner_model']} | {record['reason']}")

In [ ]:
py("scripts/summarize_results.py", "--runs", "--eval_dir", "outputs/eval", "--out_dir", "outputs/report")
py("scripts/package_outputs.py", "--source", "outputs/eval/judged", "outputs/report",
   "--zip_path", WORK / "judged_outputs.zip")
print(Path("outputs/report/report_tables.md").read_text(encoding="utf-8"))

In [ ]:
# Pairs the judge never resolved are reported here, not raised: the judgments obtained so far have to be saved
# with this version so that a new version can reuse them and retry only the unresolved pairs.
unresolved = {}
for comparison in REQUIRED:
    summary = json.loads((JUDGED_DIR / f"{comparison}_judged.summary.json").read_text(encoding="utf-8"))
    if summary["failures"] or summary["not_attempted"]:
        unresolved[comparison] = {"failures": summary["failures"], "not_attempted": summary["not_attempted"]}
if unresolved:
    print("NOTEBOOK 03 FINISHED WITH UNRESOLVED PAIRS:", unresolved)
    print("To retry only those: attach this saved version to a new version of this notebook (Add Input) and run it.")
else:
    print("NOTEBOOK 03 FINISHED. Every pair of the required comparisons has a judgment:", ", ".join(REQUIRED))

With **Save & Run All (Commit)** the saved version keeps `outputs/eval/judged/*`: judgments, `*.failures.jsonl`,
`*.attempts.jsonl` and `*.summary.json`. Unresolved pairs stay listed in the failures file and are counted
separately in every summary.